```{tip}
**For the presenter:** before starting the slideshow, run *Kernel → Restart Kernel and Run All Cells*, so all widgets are shown.
```

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
# Load the tracer reference and the prepared porosity comparison.
%run ../resources/optimization_intro.py
%run ../resources/porosity_scan.py
%run ../resources/optimization_kpi.py
porosity_data = load_porosity_measurement()
# Prepare the porosity slider and its curve-comparison scores.
porosity_scan = PorosityScanWidget(porosity_demo_configuration().process, porosity_data)
# Prepare the comparison of operating settings and their performance.
productivity_comparison = ProductivityComparison()

# 04.1 Optimization Essentials

*Introduction to chapter 04*

```{note}
This is not a lesson of its own but the introduction to the chapter: it gives the background and vocabulary used in the following lessons. It has no exercises; the methods are applied from **04.2 Parameter Fitting** onwards.
```

## Contents

**Tools:** CADET-Process (Python); interactive porosity comparison and process-performance plots.

**Python used:** Prepared model and visualization calls.

**Introduces:** parameter estimation versus process optimization; data, model parameters and variables; objectives, bounds, constraints, objective direction, Pareto fronts and convergence; the roles of `OptimizationProblem()`, `add_evaluation_object()`, `add_variable()`, `add_objective()`, `evaluate_objectives()` and `optimize()`.

1. Fit a model or improve a process
2. Variables, objectives, bounds and constraints
3. Example 1: Parameter Estimation
4. Example 2: Process Optimization
5. Competing objectives and Pareto fronts
6. When is a search finished?
7. Connect the concepts to CADET-Process

A useful model describes the experiments first. We can then use it to choose operating conditions for a separation. The lessons of this chapter follow that order: fit parameters and combine the fits into a characterization (04.2), choose collection windows (04.3) and optimize the process (04.4).

## Two kinds of optimization problems

In our chromatography examples, we focus on two kinds of optimization problems:

**Parameter estimation:** which porosity, dispersion or binding parameters make the model reproduce our measurements?

**Process optimization:** with that model, which feed duration, gradient or cycle time improves the separation?

Both need a clear definition of what can change, what should improve and which restrictions apply.

```{note}
CADET-Process provides one problem interface for different optimization algorithms. It connects simulations, comparison metrics and fractionation, reuses cached evaluations and supports competing objectives. The examples here introduce its concepts; detailed fitting and process-design workflows follow in their own lessons. A direct calculation or a simple SciPy optimizer can be sufficient for a small problem.
```

## Define a chromatography optimization

Suppose we want to collect more target protein per stationary-phase volume and time, while keeping the product sufficiently pure.

| Term | Meaning | Example |
|---|---|---|
| **Data** | Measured values that a model should reproduce | A tracer signal recorded at the column outlet |
| **Model parameters** | Values in the model equations | Column porosity, axial dispersion, binding capacity |
| **Variables** | Values the optimizer can change: selected model parameters or operating settings | Feed duration and cycle time |
| **Objective** | A calculated score to minimize or maximize | Maximize target productivity |
| **Bounds** | Allowed range of each variable | Feed duration between **20 and 90 s** |
| **Constraints** | Additional requirements a candidate must satisfy | Product purity **≥ 95%**; feeding ends before the next cycle |

Most model parameters stay fixed; only the ones registered as variables change during a search. In parameter estimation, the variables are model parameters and the data are the reference. In process optimization, the variables are operating settings and the model parameters stay fixed.

A **candidate** is one choice of variable values, such as a **60 s feed** in a **10 min cycle**. The model predicts its chromatogram; the collected fractions determine **productivity** (collected target per stationary-phase volume and time) and **purity** (the share of target in the collected product). The [01.5 Glossary](../../01%20Introduction/01.5%20Glossary/01.5a_glossary_lesson.ipynb) defines these and the other performance figures.

```{note}
Bounds and constraints define which candidates are acceptable. The objective tells us which acceptable candidate is better. A setting that collects more target protein per stationary-phase volume and time still fails the requirements if product purity is below **95%**.
```

`OptimizationProblem` stores this definition. An **optimizer** chooses candidates, evaluates them and searches for improvement.

## Reference data

Tracer is injected at **1 mol/m³ for 20 s**, followed by tracer-free liquid. At **6 mL/min**, the pulse travels through the **10 cm × 2 cm column**. The detector records the tracer concentration leaving the column for **5 min**. This experiment probes transport without binding.

```{note}
For this demonstration, the reference curve is **synthetic**: a CADET simulation generates the column-outlet signal, with no added noise or instrument delay. We treat that curve as the measurement and compare it with models at different porosities.
```

The supplied [CSV dataset](../resources/column_porosity_pulse.csv) contains **time in seconds** and **tracer concentration in mol/m³**, sampled every **0.5 s**. The [generation script](../resources/generate_intro_data.py) records how it was created. In a lab workflow, an exported detector measurement would take its place after calibration to concentration.

In [ ]:
# Show the reference curve before comparing it with a candidate model.
fig, ax = plt.subplots(figsize=(7, 3))
# Columns: 0 = time / s, 1 = tracer concentration / mol/m³
ax.plot(porosity_data[:, 0] / 60, porosity_data[:, 1], color="black")
ax.set(xlabel="Time / min", ylabel="Tracer concentration / mol/m³")
fig.tight_layout()
plt.show()

## Example 1: parameter estimation

**Objective:** minimize the fitting error between the simulated tracer curve and the reference data.

**Variable:** column porosity $\varepsilon_t$, between **0.20 and 0.60**. For each candidate porosity, compare the simulated and measured concentrations at the sampled times $t_k$:

$$
J(\varepsilon_t)=\sum_k\left[c_\mathrm{sim}(t_k;\varepsilon_t)-c_\mathrm{ref}(t_k)\right]^2
\quad\longrightarrow\quad\min.
$$

Each squared concentration difference contributes to the score. **Zero** means the two curves agree at all compared times.

At **porosity 0.40**, the candidate pulse arrives later than the reference. Moving towards **0.35** brings the curves together. Geometry, flow rate and dispersion stay fixed.

The **red shading** shows concentration differences; the plot on the right shows their **sum of squares** for each porosity. A smaller score means closer agreement.

Moving **Porosity** connects each candidate curve with its fitting error.

In [ ]:
# Display the shared porosity-scan visualization.
display(porosity_scan)

## Example 2: process optimization

**Objective:** maximize productivity. **Constraint:** product purity must meet the specified minimum.

The purity requirement is a **constraint**: candidates below **95%** are unacceptable, regardless of their productivity. It does not ask for the highest possible purity; among acceptable candidates, our objective chooses the highest productivity.

**Variables:** feed duration $t_\mathrm{feed}$ and cycle time $t_\mathrm{cycle}$. The column, flow rate and fitted model parameters stay fixed. Changing these operating settings changes how much target protein is collected and how pure the product is.

The objective is to maximize **productivity**: the collected target mass per stationary-phase volume and cycle time.

$$
PR(t_\mathrm{feed},t_\mathrm{cycle})
=\frac{m_\mathrm{target}(t_\mathrm{feed},t_\mathrm{cycle})}{V_\mathrm{solid}\,t_\mathrm{cycle}}
\quad\longrightarrow\quad\max,
\qquad \mathrm{purity}\geq 0.95.
$$

For **1 mL of stationary phase**, compare three **illustrative operating points**. The left plot shows the **variables we change**; the right shows the resulting **productivity and purity**. Matching candidate numbers link the two plots. The green star marks the best acceptable candidate among these three.

Feed duration and cycle time are **decision variables**, not objectives in this example. Their plot shows the **parameter space**. We judge those settings by the resulting productivity and purity.

From candidate **1** to **3**, the feed stays at **30 s** while the cycle shortens from **10 to 8 min**. Productivity rises from **0.500 to 0.575 mg/(mL·min)** and purity remains acceptable. Candidate **2** uses a **45 s feed** in an **8 min cycle**: its productivity is higher, but **93% purity** fails the requirement.

In [ ]:
# Link each candidate's operating variables to its KPI and purity.
_ = productivity_comparison.plot()

```{note}
The **objective** ranks acceptable settings; the **constraint** defines what is acceptable. For a positive KPI such as productivity, use `minimize=False` to maximize it. For a fitting error, smaller is better, so use `minimize=True`.
```

## Competing objectives: the pareto front

Suppose we now want **both higher productivity and higher purity**. A **Pareto front** contains settings where improving one objective requires accepting a worse value of another. The smooth curve illustrates this trade-off when many operating settings are available; the numbered points are our earlier examples.

The front is shown in **objective space**, using the scores we want to improve. Here those objectives compete. Multiple objectives need not always conflict; if one outcome is best in all objectives, the front can reduce to that single outcome.

The background points and smooth curve are **illustrative**, rather than additional CADET results. Points below the front are **dominated**: another setting gives at least the same purity and productivity, with an improvement in one or both.

A fourth illustrative setting (**45 s feed, 10 min cycle**) gives productivity **0.540 mg/(mL·min)** at **95% purity**. Candidate **3** improves both scores, so candidate **4** is **dominated** and lies off the front.

In [ ]:
# Show competing objectives and how the purity constraint changes the acceptable front.
_ = plot_productivity_pareto(productivity_comparison)

```{note}
**Constraints still apply when there are multiple objectives.** The **95% purity constraint** excludes the left part of the front, including candidate **2**. The green curve contains the acceptable trade-offs. If productivity is the only objective, the star marks the best point on this illustrative curve. If both objectives matter, the final choice depends on how we value purity against productivity. Candidate **3** remains the best acceptable choice among the original three operating points.
```

## When is a search finished?

An optimizer evaluates candidates step by step. The search has **converged** when further steps no longer improve the objective noticeably. Optimizers stop either at such a convergence criterion or when an **iteration limit** is reached; reaching the limit only means the budget ran out.

| Check | What it shows |
|---|---|
| Convergence plot of the objective over the iterations | whether the score is still falling |
| Comparison of the selected result with data or chromatogram | whether the result is useful |
| Variables at their bounds | whether a bound limits the search |

Two other kinds of convergence appear in the workshop: a cyclic process **converges to cyclic stationarity** when successive cycles give the same chromatogram (04.4), and a simulation is **grid-converged** when a finer discretization no longer changes it ([99.6 Common Problems](../../99%20Misc/99.6%20Common%20Problems/99.6a_common_problems_lesson.ipynb)). The [01.5 Glossary](../../01%20Introduction/01.5%20Glossary/01.5a_glossary_lesson.ipynb) lists these terms.

## Connect the concepts to CADET-Process

The same problem interface supports both examples:

| Call | Role |
|---|---|
| `OptimizationProblem()` | Store the problem definition |
| `add_evaluation_object(process)` | Register the process to evaluate |
| `add_variable()` | Select the process parameter or operating setting that may change, and its bounds |
| `add_objective()` | Register the score and whether to minimize or maximize it |
| `evaluate_objectives()` | Check the score for one candidate |
| `optimize()` | Let an optimizer search for better candidates |

For fitting, a candidate changes a model parameter and the score measures disagreement with data. For process design, it changes operating settings and the score measures process performance.

## How a candidate gets its score

The optimizer proposes values for the variables. To judge that candidate, we use the process model to predict a chromatogram, then calculate the objective from that prediction.

| Example | Candidate to evaluate | How we calculate its score |
|---|---|---|
| **Parameter estimation** | Set column porosity to **0.40** | Simulate the tracer curve, compare its concentrations with the reference and sum the squared differences. The score is the **fitting error**. |
| **Process optimization** | Feed duration **30 s**, cycle time **8 min** | The simulated separation gives the target amount in the collected fractions. Divided by stationary-phase volume and cycle time, it gives **productivity**; product **purity** is evaluated alongside. |

The optimizer uses these results to choose new candidates: seek a **smaller fitting error**, or **higher productivity while meeting the purity constraint**. Repeating this evaluation is the optimization search.

In this lesson, the tracer curves are prepared CADET simulations. The process example uses illustrative performance values to explain the choice. The following lessons build the evaluations that calculate these scores from measurements and simulated product fractions.

## Summary

- Data are the reference; model parameters describe the system; variables are the values an optimizer changes.
- Variables define what changes; objectives define what should improve.
- Bounds and constraints define acceptable candidates.
- Fitting compares simulated and measured curves; process optimization evaluates collected products.
- Several objectives can lead to Pareto trade-offs, with constraints still applying.
- A search has converged when the objective stops improving; an iteration limit alone is not convergence.

**Next lesson:** [**04.2 Parameter Fitting**](../04.2%20Parameter%20Fitting/04.2a_parameter_fitting_lesson.ipynb) connects measurements, simulations and a fitting objective.

Detailed mathematical examples and solver comparisons are available in [99.4 Optimization Basics](../../99%20Misc/99.4%20Optimization%20Basics/99.4a_optimization_basics_lesson.ipynb). Evaluator details and additional techniques are in [99.5 Advanced Optimization](../../99%20Misc/99.5%20Advanced%20Optimization/99.5a_advanced_optimization_lesson.ipynb).